In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import numpy as np

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns



###########################################
#import the csv into jupyter notebook
#the dataset
############################################
merged_df1 = pd.DataFrame()
# 循环读取并合并18个数据集
for i in range(501, 519):
    # 读取数据集
    df = pd.read_csv(f'/root/Project/har70dataset/{i}.csv')
    # 第一个数据集直接赋值给merged_df
    if i == 501:
        merged_df1 = df
    else:
        # 根据共同列进行合并
        merged_df1 = pd.concat([merged_df1,df],ignore_index=True)
# 输出合并后的数据集
print(merged_df1)



###########################
#data preprocessing
###########################
#降噪
# 创建数据副本
merged_df = merged_df1.copy()
# 设置移动均值滤波的窗口大小
N = 5
# 指定要过滤的加速度数据列
columns_to_filter = ['back_x', 'back_y', 'back_z', 'thigh_x', 'thigh_y', 'thigh_z']
# 对指定的列应用移动均值滤波，并直接更新这些列
for col in columns_to_filter:
    merged_df[col] = merged_df[col].rolling(window=N).mean()
# 删除移动均值滤波导致的前N-1行空值
merged_df = merged_df.iloc[N-1:].reset_index(drop=True)


# 分离特征和标签
X = merged_df.drop(columns=['label','timestamp']).values   #transform the dataframe into numpy
y = merged_df['label'].values   #transform the dataframe into numpy


MLP

In [ ]:
# 将标签转换为one-hot编码
y = to_categorical(y)

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 创建MLP模型
model = Sequential([
    Dense(64, input_dim=X_train.shape[1], activation='relu'),
    Dense(64, activation='relu'),
    Dense(y_train.shape[1], activation='softmax')
])

# 编译模型
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 训练模型
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2)

# 预测测试集
y_pred = model.predict(X_test)

# 由于预测结果是one-hot编码的形式，需要转换为标签
y_test_labels = np.argmax(y_test, axis=1)
y_pred_labels = np.argmax(y_pred, axis=1)

# 计算总体指标并打印
loss, accuracy = model.evaluate(X_test, y_test)
print(f'测试集上的损失: {loss:.4f}, 准确率: {accuracy:.4f}')

# 使用classification_report打印每个标签的精确度、召回率、F1分数
report = classification_report(y_test_labels, y_pred_labels, target_names=['Label1', 'Label3', 'Label4','Label5','Label6','Label7','Label8'])
print(report)
# 生成混淆矩阵
cm = confusion_matrix(y_test_labels, y_pred_labels)

# 计算并打印每个标签的准确率
for i, label in enumerate(cm.diagonal()):
    label_accuracy = label / cm.sum(axis=1)[i]
    print(f'标签 {i} 的准确率: {label_accuracy:.4f}')


# 计算loss，accuracy
loss, accuracy = model.evaluate(X_test, y_test)
print(f'测试集上的损失: {loss:.4f}, 准确率: {accuracy:.4f}')
# 计算Precision
precision = precision_score(y_test_labels, y_pred_labels, average='macro')
print(f'总体Precision: {precision:.4f}')
# 计算Recall
recall = recall_score(y_test_labels, y_pred_labels, average='macro')
print(f'总体Recall: {recall:.4f}')
# 计算F1分数
f1 = f1_score(y_test_labels, y_pred_labels, average='macro')
print(f'总体F1分数: {f1:.4f}')
# 生成混淆矩阵
cm = confusion_matrix(y_test_labels, y_pred_labels)
print('混淆矩阵:')
print(cm)


# 将混淆矩阵正则化，转换为百分比
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# 设置图表大小
plt.figure(figsize=(10, 8))

# 用Seaborn画热力图
ax = sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues')

# 设置标题和标签
ax.set_title('Confusion matrix (normalized to % of total test data)')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')

# 为每个轴设置刻度标签
ax.xaxis.set_ticklabels(['1', '3', '4', '5', '6', '7', '8'])
ax.yaxis.set_ticklabels(['1', '3', '4', '5', '6', '7', '8'])

# 显示图形
plt.show()
plt.savefig('/root/Project/MLP.png')


CNN

In [ ]:
# 将标签转换为one-hot编码
y = to_categorical(y)
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=15)
# 重构数据以适应CNN的输入
X_train = X_train.reshape((X_train.shape[0], 2, 3, 1))
X_test = X_test.reshape((X_test.shape[0], 2, 3, 1))
# 构建CNN模型
model = models.Sequential([
    layers.ZeroPadding2D(padding=(1, 1), input_shape=(2, 3, 1)),  # 加入零填充层
    layers.Conv2D(32, (2, 2), activation='relu'),  # 共32个卷积同时运算，激活函数为relu
    layers.MaxPooling2D((1, 1)),  # 其余层维持不变
    layers.Flatten(), #展平为一维向量
    layers.Dense(64, activation='relu'), #全连接层的神经元为64个
    layers.Dense(y.shape[1], activation='softmax')  # 输出层，用于分类
])
# 编译模型
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# 训练模型
model.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test))
# 预测测试集
y_pred = model.predict(X_test)
# 由于预测结果是one-hot编码的形式，需要转换为标签
y_test_labels = np.argmax(y_test, axis=1)
y_pred_labels = np.argmax(y_pred, axis=1)
# 计算loss，accuracy
# 计算总体指标并打印
loss, accuracy = model.evaluate(X_test, y_test)
print(f'测试集上的损失: {loss:.4f}, 准确率: {accuracy:.4f}')

# 使用classification_report打印每个标签的精确度、召回率、F1分数
report = classification_report(y_test_labels, y_pred_labels, target_names=['Label1', 'Label3', 'Label4','Label5','Label6','Label7','Label8'])
print(report)
# 生成混淆矩阵
cm = confusion_matrix(y_test_labels, y_pred_labels)

# 计算并打印每个标签的准确率
for i, label in enumerate(cm.diagonal()):
    label_accuracy = label / cm.sum(axis=1)[i]
    print(f'标签 {i} 的准确率: {label_accuracy:.4f}')


# 计算loss，accuracy
loss, accuracy = model.evaluate(X_test, y_test)
print(f'测试集上的损失: {loss:.4f}, 准确率: {accuracy:.4f}')
# 计算Precision
precision = precision_score(y_test_labels, y_pred_labels, average='macro')
print(f'总体Precision: {precision:.4f}')
# 计算Recall
recall = recall_score(y_test_labels, y_pred_labels, average='macro')
print(f'总体Recall: {recall:.4f}')
# 计算F1分数
f1 = f1_score(y_test_labels, y_pred_labels, average='macro')
print(f'总体F1分数: {f1:.4f}')
# 生成混淆矩阵
cm = confusion_matrix(y_test_labels, y_pred_labels)
print('混淆矩阵:')
print(cm)


# 将混淆矩阵正则化，转换为百分比
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# 设置图表大小
plt.figure(figsize=(10, 8))

# 用Seaborn画热力图
ax = sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues')

# 设置标题和标签
ax.set_title('Confusion matrix (normalized to % of total test data)')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')

# 为每个轴设置刻度标签
ax.xaxis.set_ticklabels(['1', '3', '4', '5', '6', '7', '8'])
ax.yaxis.set_ticklabels(['1', '3', '4', '5', '6', '7', '8'])

# 显示图形
plt.show()
plt.savefig('/root/Project/CNN.png')


LSTM 

In [ ]:
# 分离特征和标签
X = merged_df.drop(columns=['label','timestamp']).values
y = merged_df['label'].values

# 添加时间步长维度
X = np.expand_dims(X, axis=1)

# 将标签编码为整数
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 将整数标签转换为one-hot编码
y_onehot = to_categorical(y_encoded)

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=0.3, random_state=42)

# 创建LSTM模型

#线性堆叠层.两层LSTM将按顺序执行，其中每层的输入都来自前一层的输出
model = Sequential()
#第一个LSTM层，激活函数使用默认
model.add(LSTM(128, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True))
model.add(Dropout(0.5))
#第二个LSTM层
model.add(LSTM(64, return_sequences=False ))
model.add(Dropout(0.5))


#加了全连接层Dense，用于将加速度数据分类
model.add(Dense(y_train.shape[1], activation='softmax'))


# 编译模型
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 添加早停以防止过拟合
early_stopping = EarlyStopping(monitor='val_loss', patience=5)

# 训练模型
model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.2, callbacks=[early_stopping])

# 预测测试集
y_pred = model.predict(X_test)
# 由于预测结果是one-hot编码的形式，需要转换为标签
y_test_labels = np.argmax(y_test, axis=1)
y_pred_labels = np.argmax(y_pred, axis=1)
# 计算loss，accuracy
# 计算总体指标并打印
loss, accuracy = model.evaluate(X_test, y_test)
print(f'测试集上的损失: {loss:.4f}, 准确率: {accuracy:.4f}')

# 使用classification_report打印每个标签的精确度、召回率、F1分数
report = classification_report(y_test_labels, y_pred_labels, target_names=['Label1', 'Label3', 'Label4','Label5','Label6','Label7','Label8'])
print(report)
# 生成混淆矩阵
cm = confusion_matrix(y_test_labels, y_pred_labels)

# 计算并打印每个标签的准确率
for i, label in enumerate(cm.diagonal()):
    label_accuracy = label / cm.sum(axis=1)[i]
    print(f'标签 {i} 的准确率: {label_accuracy:.4f}')


# 计算loss，accuracy
loss, accuracy = model.evaluate(X_test, y_test)
print(f'测试集上的损失: {loss:.4f}, 准确率: {accuracy:.4f}')
# 计算Precision
precision = precision_score(y_test_labels, y_pred_labels, average='macro')
print(f'总体Precision: {precision:.4f}')
# 计算Recall
recall = recall_score(y_test_labels, y_pred_labels, average='macro')
print(f'总体Recall: {recall:.4f}')
# 计算F1分数
f1 = f1_score(y_test_labels, y_pred_labels, average='macro')
print(f'总体F1分数: {f1:.4f}')
# 生成混淆矩阵
cm = confusion_matrix(y_test_labels, y_pred_labels)
print('混淆矩阵:')
print(cm)


# 将混淆矩阵正则化，转换为百分比
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# 设置图表大小
plt.figure(figsize=(10, 8))

# 用Seaborn画热力图
ax = sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues')

# 设置标题和标签
ax.set_title('Confusion matrix (normalized to % of total test data)')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')

# 为每个轴设置刻度标签
ax.xaxis.set_ticklabels(['1', '3', '4', '5', '6', '7', '8'])
ax.yaxis.set_ticklabels(['1', '3', '4', '5', '6', '7', '8'])

# 显示图形
plt.show()
plt.savefig('/root/Project/LSTM.png')
